# Image Data Augmentation

Basado en el diagrama de Shorten & Khoshgoftaar (2019), *A survey on Image Data Augmentation for Deep Learning*, Journal of Big Data.

Estructura:
- Basic Image Manipulations: Geometric Transformations, Kernel Filters, Color Space Transformations, Random Erasing, Mixing Images
- Deep Learning Approaches: Adversarial Training, Neural Style Transfer, GAN Data Augmentation
- Meta Learning: AutoAugment, RandAugment, Smart Augmentation

In [2]:
# Celda 0 - Instalacion de dependencias (solo necesario en Colab)
!pip install albumentations opencv-python-headless matplotlib pillow torchvision scikit-image -q

^C


In [ ]:
# SECCIÓN 1: Imports y Configuración de I/O Local
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tkinter import Tk, filedialog
import warnings
warnings.filterwarnings('ignore')

# 1. Definir la ruta absoluta de salida en Laragon (Windows Raw String)
OUTPUT_DIR = r"C:\laragon\www\IA\GIT\IAM_h2_dataset_ilupica\MERCIER_MATIAS\data_augmentation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Módulo de ingesta dinámica de imágenes
def load_input_images():
    # Inicializar y ocultar la ventana base de Tkinter
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True) # Fuerza la ventana al frente
    
    print("Abriendo explorador de archivos... Selecciona tus 5 imágenes.")
    file_paths = filedialog.askopenfilenames(
        title="Selecciona 5 imágenes base para el Data Augmentation",
        filetypes=[("Image files", "*.jpg *.jpeg *.png *.webp")]
    )
    
    if len(file_paths) != 5:
        print(f"⚠️ Alerta: Seleccionaste {len(file_paths)} imágenes. El pipeline espera 5 para generar 50 exactas.")
        
    images_data = []
    for path in file_paths:
        img_pil = Image.open(path).convert('RGB')
        img_pil = img_pil.resize((256, 256))
        
        # Guardar tupla con el nombre base y el tensor numpy
        base_filename = os.path.splitext(os.path.basename(path))[0]
        images_data.append((base_filename, np.array(img_pil)))
        
    return images_data

# Ejecutar carga en memoria
input_images = load_input_images()
print(f" {len(input_images)} imágenes cargadas en el buffer de memoria.")

## Basic Image Manipulations
### 1. Geometric Transformations

In [ ]:
# Geometric Transformations
# Rotacion, flip, escala, perspectiva, traslacion

def apply_geometric_transformations(image):
    """Aplica transformaciones geométricas básicas y empaqueta en un diccionario."""
    results = {}
    results['1_geom_flip_h']  = A.HorizontalFlip(p=1.0)(image=image)['image']
    results['1_geom_flip_v']  = A.VerticalFlip(p=1.0)(image=image)['image']
    results['1_geom_rotate']  = A.Rotate(limit=(-30, 30), p=1.0)(image=image)['image']
    results['1_geom_escala']  = A.ShiftScaleRotate(shift_limit=0.0, scale_limit=0.35, rotate_limit=0, p=1.0)(image=image)['image']
    results['1_geom_trasla']  = A.ShiftScaleRotate(shift_limit=0.15, scale_limit=0.0, rotate_limit=0, p=1.0)(image=image)['image']
    results['1_geom_persp']   = A.Perspective(scale=(0.05, 0.12), p=1.0)(image=image)['image']
    results['1_geom_elastic'] = A.ElasticTransform(alpha=60, sigma=6, p=1.0)(image=image)['image']
    return results

# Ejecución local para renderizar el output visual en el notebook
geom_results = apply_geometric_transformations(img_rgb)

mostrar(
    [img_rgb,
     geom_results['1_geom_flip_h'], geom_results['1_geom_flip_v'],
     geom_results['1_geom_rotate'], geom_results['1_geom_escala'],
     geom_results['1_geom_trasla'], geom_results['1_geom_persp'],
     geom_results['1_geom_elastic']],
    ['Original', 'Flip Horizontal', 'Flip Vertical', 'Rotacion', 'Escala', 'Traslacion', 'Perspectiva', 'Elastic Transform']
)

### 2. Kernel Filters

In [ ]:
# Kernel Filters
# Desenfoque, nitidez, Sobel, Laplaciano, Emboss

def apply_kernel_filters(image):
    """Aplica filtros de convolución espacial y empaqueta en un diccionario."""
    results = {}

    # Desenfoques
    results['2_filter_blur_g'] = cv2.GaussianBlur(image, (7, 7), 0)
    results['2_filter_blur_m'] = cv2.medianBlur(image, 5)

    # Nitidez (Sharpening)
    kernel_sharp = np.array([[ 0, -1,  0],
                             [-1,  5, -1],
                             [ 0, -1,  0]])
    results['2_filter_sharp'] = cv2.filter2D(image, -1, kernel_sharp)

    # Extracción de bordes (Conversión temporal a escala de grises necesaria)
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Sobel
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    results['2_filter_sobel'] = np.uint8(np.clip(np.sqrt(sobelx**2 + sobely**2), 0, 255))

    # Laplaciano
    results['2_filter_laplacian'] = np.uint8(np.clip(np.abs(cv2.Laplacian(gray, cv2.CV_64F)), 0, 255))

    # Emboss (Relieve)
    kernel_emboss = np.array([[-2, -1, 0],
                              [-1,  1, 1],
                              [ 0,  1, 2]])
    results['2_filter_emboss'] = cv2.filter2D(image, -1, kernel_emboss)

    return results

# Ejecución local para renderizar el output visual en el notebook
filter_results = apply_kernel_filters(img_rgb)

mostrar(
    [img_rgb,
     filter_results['2_filter_blur_g'], filter_results['2_filter_blur_m'],
     filter_results['2_filter_sharp'], filter_results['2_filter_sobel'],
     filter_results['2_filter_laplacian'], filter_results['2_filter_emboss']],
    ['Original', 'Blur Gaussiano', 'Blur Mediana', 'Nitidez', 'Sobel', 'Laplaciano', 'Emboss']
)

### 3. Color Space Transformations

In [ ]:
# Color Space Transformations
# Brillo, contraste, saturacion, hue, CLAHE, escala de grises, canales

def apply_color_space(image):
    """Aplica alteraciones en el espacio de color y empaqueta en un diccionario."""
    results = {}

    # Ajustes básicos
    results['3_color_bright']   = A.RandomBrightnessContrast(brightness_limit=(0.4, 0.4), contrast_limit=0, p=1.0)(image=image)['image']
    results['3_color_contrast'] = A.RandomBrightnessContrast(brightness_limit=0, contrast_limit=(0.5, 0.5), p=1.0)(image=image)['image']
    results['3_color_hue_sat']  = A.HueSaturationValue(hue_shift_limit=40, sat_shift_limit=60, val_shift_limit=0, p=1.0)(image=image)['image']

    # Mejora de histograma y Gamma
    results['3_color_clahe']    = A.CLAHE(clip_limit=6.0, tile_grid_size=(8, 8), p=1.0)(image=image)['image']
    results['3_color_gamma']    = A.RandomGamma(gamma_limit=(40, 40), p=1.0)(image=image)['image']

    # Conversión a HSV y visualización
    img_hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    results['3_color_hsv_vis']  = cv2.cvtColor(img_hsv, cv2.COLOR_HSV2RGB)

    # Escala de grises mapeada a 3 canales (para mantener consistencia tensorial/matricial posterior)
    results['3_color_gray_3ch'] = cv2.cvtColor(cv2.cvtColor(image, cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)

    return results

# Ejecución local para renderizar el output visual en el notebook
color_results = apply_color_space(img_rgb)

mostrar(
    [img_rgb,
     color_results['3_color_bright'], color_results['3_color_contrast'],
     color_results['3_color_hue_sat'], color_results['3_color_clahe'],
     color_results['3_color_gamma'], color_results['3_color_hsv_vis'],
     color_results['3_color_gray_3ch']],
    ['Original', 'Brillo+', 'Contraste+', 'Hue-Saturacion', 'CLAHE', 'Gamma', 'Espacio HSV', 'Escala de grises']
)

### 4. Random Erasing (Cutout / CoarseDropout)

In [ ]:
# Random Erasing (Cutout / CoarseDropout)
# Cutout (1 region), CoarseDropout (multiples regiones), GridDropout

def apply_random_erasing(image):
    """Aplica técnicas de borrado aleatorio (Random Erasing) y empaqueta en un diccionario."""
    results = {}

    # Cutout clásico (1 solo hueco grande)
    results['4_erasing_cutout'] = A.CoarseDropout(
        max_holes=1, max_height=80, max_width=80,
        min_holes=1, min_height=60, min_width=60,
        fill_value=0, p=1.0
    )(image=image)['image']

    # CoarseDropout (múltiples huecos grises)
    results['4_erasing_coarse'] = A.CoarseDropout(
        max_holes=10, max_height=30, max_width=30,
        min_holes=5,  min_height=15, min_width=15,
        fill_value=128, p=1.0
    )(image=image)['image']

    # CoarseDropout a color (fill con valor RGB)
    results['4_erasing_coarse_color'] = A.CoarseDropout(
        max_holes=8, max_height=25, max_width=25,
        min_holes=4, min_height=10, min_width=10,
        fill_value=[255, 0, 0], p=1.0
    )(image=image)['image']

    # GridDropout (caída de píxeles en formato cuadrícula)
    results['4_erasing_grid'] = A.GridDropout(ratio=0.35, p=1.0)(image=image)['image']

    return results

# Ejecución local para renderizar el output visual en el notebook
erasing_results = apply_random_erasing(img_rgb)

mostrar(
    [img_rgb,
     erasing_results['4_erasing_cutout'], erasing_results['4_erasing_coarse'],
     erasing_results['4_erasing_coarse_color'], erasing_results['4_erasing_grid']],
    ['Original', 'Cutout (1 region)', 'CoarseDropout gris', 'CoarseDropout color', 'GridDropout']
)

### 5. Mixing Images (MixUp y CutMix)

In [ ]:
# Se utilizan dos imagenes distintas

def apply_mixing_images(image):
    """Aplica técnicas de MixUp y CutMix combinando la imagen de entrada con una secundaria."""
    results = {}

    # Encapsulamos el I/O de la segunda imagen para no romper la firma de la función.
    # (Nota: El link original descarga el mismo perro, aunque el label dice 'gato').
    url2 = 'https://cdn.pixabay.com/photo/2026/03/23/07/20/salofoto-dog-10187835_960_720.jpg'
    response2 = requests.get(url2)
    img2_pil = Image.open(BytesIO(response2.content)).convert('RGB').resize((256, 256))
    img2_rgb  = np.array(img2_pil)

    # Guardamos la imagen target en el diccionario para el export
    results['5_mixing_target_B'] = img2_rgb

    # MixUp
    alpha = 0.4
    lam   = np.random.beta(alpha, alpha)
    results['5_mixup'] = np.clip(
        lam * image.astype(np.float32) + (1 - lam) * img2_rgb.astype(np.float32), 0, 255
    ).astype(np.uint8)

    # CutMix (Helper function aislada en el scope local)
    def get_cutmix(img_a, img_b, alpha_c=0.4):
        result = img_a.copy()
        h, w = result.shape[:2]
        lam_c = np.random.beta(alpha_c, alpha_c)
        cut_h = int(h * np.sqrt(1 - lam_c))
        cut_w = int(w * np.sqrt(1 - lam_c))
        cx = np.random.randint(0, w)
        cy = np.random.randint(0, h)
        x1 = np.clip(cx - cut_w // 2, 0, w)
        x2 = np.clip(cx + cut_w // 2, 0, w)
        y1 = np.clip(cy - cut_h // 2, 0, h)
        y2 = np.clip(cy + cut_h // 2, 0, h)
        result[y1:y2, x1:x2] = img_b[y1:y2, x1:x2]
        return result

    results['5_cutmix_a_b'] = get_cutmix(image, img2_rgb)
    results['5_cutmix_b_a'] = get_cutmix(img2_rgb, image)

    return results

# Ejecución local para renderizar el output visual en el notebook
mixing_results = apply_mixing_images(img_rgb)

mostrar(
    [img_rgb, mixing_results['5_mixing_target_B'], mixing_results['5_mixup'],
     mixing_results['5_cutmix_a_b'], mixing_results['5_cutmix_b_a']],
    ['Imagen A (perro)', 'Imagen B (gato)', 'MixUp', 'CutMix A->B', 'CutMix B->A']
)

## Deep Learning Approaches
### 6. Adversarial Training (FGSM)

In [ ]:
# Genera ejemplos adversariales para entrenar modelos mas robustos

def apply_adversarial_fgsm(image):
    """Genera ejemplos adversariales usando FGSM y los empaqueta en un diccionario."""
    results = {}

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Instanciamos el modelo localmente para mantener la función independiente
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)
    model.eval()

    # Convertimos el array de NumPy a PIL Image para que T.Resize funcione correctamente
    img_pil_input = Image.fromarray(image)

    preprocess = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    tensor_input = preprocess(img_pil_input).unsqueeze(0).to(device)
    tensor_input.requires_grad_(True)

    output = model(tensor_input)
    pred_class = output.argmax(dim=1)

    # Vaciamos gradientes previos por buena práctica antes del backward pass
    model.zero_grad()
    loss = nn.CrossEntropyLoss()(output, pred_class)
    loss.backward()

    def deprocess(t):
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img_out = (t.detach().cpu().squeeze() * std + mean).clamp(0, 1)
        return (img_out.permute(1, 2, 0).numpy() * 255).astype(np.uint8)

    epsilons = [0.05, 0.2, 0.5, 1]

    for eps in epsilons:
        perturbed = tensor_input + eps * tensor_input.grad.sign()
        # Se guarda el resultado en el diccionario
        results[f'6_fgsm_eps_{eps}'] = deprocess(perturbed)

    return results

# Ejecución local para renderizar el output visual en el notebook
fgsm_results = apply_adversarial_fgsm(img_rgb)

# Preparar las listas iterando sobre el diccionario para la función mostrar()
epsilons = [0.05, 0.2, 0.5, 1]
imagenes_a_mostrar = [img_rgb] + [fgsm_results[f'6_fgsm_eps_{e}'] for e in epsilons]
titulos_a_mostrar = ['Original'] + [f'FGSM eps={e}' for e in epsilons]

mostrar(imagenes_a_mostrar, titulos_a_mostrar)

In [ ]:
# 6.1 Visualización del ruido FGSM

def extract_fgsm_noise(image):
    """Extrae y normaliza el ruido adversarial generado por FGSM en un diccionario."""
    results = {}

    # Re-inicialización y recálculo de gradientes para mantener la función pura
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)
    model.eval()

    img_pil_input = Image.fromarray(image)
    preprocess = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    tensor_input = preprocess(img_pil_input).unsqueeze(0).to(device)
    tensor_input.requires_grad_(True)

    output = model(tensor_input)
    pred_class = output.argmax(dim=1)

    model.zero_grad()
    loss = nn.CrossEntropyLoss()(output, pred_class)
    loss.backward()

    epsilons = [0.05, 0.2, 0.5, 1]
    eps_max = max(epsilons)

    # Función de utilidad interna
    def get_ruido(grad, eps, max_e):
        ruido = grad.detach().cpu().squeeze().permute(1, 2, 0).numpy() * eps
        # Normalizar todos los paneles con el mismo rango
        ruido = ((ruido + max_e) / (2 * max_e) * 255).clip(0, 255).astype(np.uint8)
        return ruido

    for eps in epsilons:
        results[f'6_fgsm_noise_eps_{eps}'] = get_ruido(tensor_input.grad, eps, eps_max)

    return results

# Ejecución local para renderizar el output visual en el notebook
noise_results = extract_fgsm_noise(img_rgb)

epsilons = [0.05, 0.2, 0.5, 1]
imagenes_ruido = [noise_results[f'6_fgsm_noise_eps_{e}'] for e in epsilons]
titulos_ruido = [f'Ruido eps={e}' for e in epsilons]

mostrar(imagenes_ruido, titulos_ruido)

### 7. Neural Style Transfer (NST)

In [ ]:
# Transfiere el estilo visual de una imagen a otra usando VGG19
# Implementación aislada para el pipeline

def apply_neural_style_transfer(image):
    """Aplica Neural Style Transfer usando VGG19 y empaqueta el resultado en un diccionario."""
    results = {}

    import torch.optim as optim
    from torchvision.models import vgg19, VGG19_Weights

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # ---------------------------------------------------------
    # Funciones auxiliares anidadas (Aislamiento de Scope)
    # ---------------------------------------------------------
    def imagen_a_tensor(img_array, size=256):
        transform = T.Compose([
            T.ToPILImage(),
            T.Resize((size, size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        return transform(img_array).unsqueeze(0).to(device)

    def tensor_a_imagen(tensor):
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(device)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(device)
        img  = (tensor.squeeze() * std + mean).clamp(0, 1)
        return (img.detach().cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)

    def gram_matrix(feature_map):
        b, c, h, w = feature_map.size()
        features = feature_map.view(b * c, h * w)
        return torch.mm(features, features.t()) / (b * c * h * w)

    def extraer_features(img_tensor, modelo, capas):
        features = {}
        x = img_tensor
        for idx, layer in enumerate(modelo):
            x = layer(x)
            if idx in capas:
                features[idx] = x
        return features

    # Instanciamos el modelo localmente
    vgg = vgg19(weights=VGG19_Weights.DEFAULT).features.to(device).eval()
    for param in vgg.parameters():
        param.requires_grad_(False)

    content_layer = 21   # relu4_2
    style_layers  = [0, 5, 10, 19, 28]  # relu1_1 ... relu5_1

    # Fetch de la imagen de estilo
    headers = {'User-Agent': 'Mozilla/5.0 (compatible; NST-notebook/1.0)'}
    url_style = 'https://cdn.pixabay.com/photo/2017/03/23/15/57/watercolour-2168729_1280.jpg'
    resp_s = requests.get(url_style, headers=headers, timeout=15)
    resp_s.raise_for_status()

    img_style_pil = Image.open(BytesIO(resp_s.content)).convert('RGB')
    img_style_rgb = np.array(img_style_pil.resize((256, 256)))

    # Guardamos el target de estilo para la exportación final
    results['7_nst_style_target'] = img_style_rgb

    content_tensor = imagen_a_tensor(image)
    style_tensor   = imagen_a_tensor(img_style_rgb)

    content_features = extraer_features(content_tensor, vgg, [content_layer])
    style_features   = extraer_features(style_tensor, vgg, style_layers)
    style_grams      = {l: gram_matrix(style_features[l]) for l in style_layers}

    # Inicialización del ruido con la imagen de contenido
    generated = content_tensor.clone().requires_grad_(True)
    optimizer = optim.Adam([generated], lr=0.01)

    weight_content = 1e4
    weight_style   = 1e7
    n_steps        = 1000

    # Loop de optimización
    for step in range(n_steps):
        gen_features  = extraer_features(generated, vgg, [content_layer] + style_layers)
        loss_content  = nn.MSELoss()(gen_features[content_layer], content_features[content_layer])
        loss_style    = sum(
            nn.MSELoss()(gram_matrix(gen_features[l]), style_grams[l])
            for l in style_layers
        )
        loss_total = weight_content * loss_content + weight_style * loss_style

        optimizer.zero_grad()
        loss_total.backward(retain_graph=True)
        optimizer.step()

    # Guardamos el resultado procesado
    results['7_nst_result'] = tensor_a_imagen(generated)

    return results

# Ejecución local para renderizar el output visual en el notebook
# Nota: Imprimimos un log de estado porque este bloque tiene un tiempo de cómputo considerable
print("Ejecutando iteraciones de Neural Style Transfer (esto tomará unos segundos/minutos)...")
nst_results = apply_neural_style_transfer(img_rgb)

mostrar(
    [img_rgb, nst_results['7_nst_style_target'], nst_results['7_nst_result']],
    ['Contenido (perro)', 'Estilo (Van Gogh)', 'Resultado NST']
)

### 8. GAN Data Augmentation

In [ ]:
# DCGAN entrenada desde cero en las imagenes de la clase objetivo
# Esta celda define y entrena una DCGAN minimalista sobre tiles de la imagen original

def apply_gan_data_augmentation(image):
    """Entrena una DCGAN con parches de la imagen y empaqueta las imágenes generadas."""
    results = {}

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Funciones y Clases anidadas (Aislamiento de Scope)

    def extraer_patches(img, patch_size=64, n_patches=200):
        h, w = img.shape[:2]
        patches = []
        for _ in range(n_patches):
            y = np.random.randint(0, h - patch_size)
            x = np.random.randint(0, w - patch_size)
            patch = img[y:y+patch_size, x:x+patch_size]
            patches.append(patch)
        return np.array(patches)

    class Generator(nn.Module):
        def __init__(self, latent_dim=100):
            super().__init__()
            self.net = nn.Sequential(
                nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False),  # -> 4x4
                nn.BatchNorm2d(512), nn.ReLU(True),
                nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),         # -> 8x8
                nn.BatchNorm2d(256), nn.ReLU(True),
                nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),         # -> 16x16
                nn.BatchNorm2d(128), nn.ReLU(True),
                nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),          # -> 32x32
                nn.BatchNorm2d(64),  nn.ReLU(True),
                nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),            # -> 64x64
                nn.Tanh()
            )
        def forward(self, z):
            return self.net(z)

    class Discriminator(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(3, 64, 4, 2, 1, bias=False),                     # -> 32x32
                nn.LeakyReLU(0.2, inplace=True),
                nn.Conv2d(64, 128, 4, 2, 1, bias=False),                   # -> 16x16
                nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
                nn.Conv2d(128, 256, 4, 2, 1, bias=False),                  # -> 8x8
                nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True),
                nn.Conv2d(256, 512, 4, 2, 1, bias=False),                  # -> 4x4
                nn.BatchNorm2d(512), nn.LeakyReLU(0.2, inplace=True),
                nn.Conv2d(512, 1, 4, 1, 0, bias=False),                    # -> 1x1
                nn.Sigmoid()
            )
        def forward(self, x):
            return self.net(x).view(-1)

    def gan_tensor_a_img(t):
        arr = ((t.permute(1, 2, 0).numpy() + 1) * 127.5).clip(0, 255).astype(np.uint8)
        return arr
    # ---------------------------------------------------------

    # 1. Generar dataset sintético
    patches = extraer_patches(image, patch_size=64, n_patches=300)

    # Normalizar a [-1, 1]
    patches_t = torch.tensor(patches).permute(0, 3, 1, 2).float() / 127.5 - 1.0
    from torch.utils.data import DataLoader, TensorDataset
    dataset   = TensorDataset(patches_t)
    loader    = DataLoader(dataset, batch_size=32, shuffle=True)

    # 2. Inicializar modelos y optimizadores
    LATENT_DIM = 100
    G = Generator(LATENT_DIM).to(device)
    D = Discriminator().to(device)

    criterion = nn.BCELoss()
    opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

    EPOCHS = 30

    # 3. Loop de entrenamiento
    for epoch in range(EPOCHS):
        for (real_batch,) in loader:
            real_batch = real_batch.to(device)
            batch_size = real_batch.size(0)
            real_labels = torch.ones(batch_size).to(device)
            fake_labels = torch.zeros(batch_size).to(device)

            # Entrenar Discriminador
            z = torch.randn(batch_size, LATENT_DIM, 1, 1).to(device)
            fake_imgs = G(z).detach()
            loss_D = criterion(D(real_batch), real_labels) + criterion(D(fake_imgs), fake_labels)
            opt_D.zero_grad()
            loss_D.backward()
            opt_D.step()

            # Entrenar Generador
            z = torch.randn(batch_size, LATENT_DIM, 1, 1).to(device)
            loss_G = criterion(D(G(z)), real_labels)
            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()

    # 4. Generar imágenes sintéticas finales
    G.eval()
    with torch.no_grad():
        z_samples = torch.randn(6, LATENT_DIM, 1, 1).to(device)
        fake_samples = G(z_samples).cpu()

    # 5. Empaquetar resultados
    for i in range(6):
        results[f'8_gan_generated_{i+1}'] = gan_tensor_a_img(fake_samples[i])

    return results

# Ejecución local para renderizar el output visual en el notebook
print('Entrenando DCGAN por 30 epocas (esto puede tomar varios minutos)...')
gan_results = apply_gan_data_augmentation(img_rgb)

imagenes_generadas = [img_rgb] + [gan_results[f'8_gan_generated_{i+1}'] for i in range(6)]
titulos_generados = ['Original (256x256)'] + [f'GAN generada {i+1}' for i in range(6)]

mostrar(imagenes_generadas, titulos_generados, figsize_w=3)

## Meta Learning
### 9. AutoAugment

In [ ]:
# Politica de aumento aprendida automaticamente sobre ImageNet, CIFAR-10 y SVHN
# Referencia: Cubuk et al. (2019), CVPR

def apply_autoaugment(image):
    """Aplica políticas preentrenadas de AutoAugment y empaqueta en un diccionario."""
    results = {}

    # Importaciones aisladas
    from torchvision import transforms
    from PIL import Image

    # Convertir el array de NumPy a PIL Image para torchvision
    img_pil_input = Image.fromarray(image)

    policies = {
        'ImageNet': transforms.AutoAugmentPolicy.IMAGENET,
        'CIFAR10':  transforms.AutoAugmentPolicy.CIFAR10,
        'SVHN':     transforms.AutoAugmentPolicy.SVHN
    }

    # Generar 3 variantes por cada política
    for policy_name, policy in policies.items():
        aug = transforms.AutoAugment(policy=policy)
        for i in range(3):
            augmented_pil = aug(img_pil_input)
            results[f'9_autoaugment_{policy_name}_var_{i+1}'] = np.array(augmented_pil)

    return results

# Ejecución local para renderizar el output visual en el notebook
autoaugment_results = apply_autoaugment(img_rgb)

# Renderizamos los resultados agrupados por política, igual que en tu celda original
policies_names = ['ImageNet', 'CIFAR10', 'SVHN']
for policy_name in policies_names:
    variantes = [autoaugment_results[f'9_autoaugment_{policy_name}_var_{i+1}'] for i in range(3)]
    mostrar(
        [img_rgb] + variantes,
        ['Original'] + [f'{policy_name} variante {i+1}' for i in range(3)]
    )

### 10. RandAugment y TrivialAugmentWide (Smart Augmentation)

In [ ]:
# 10. RandAugment y TrivialAugmentWide (Smart Augmentation)
# RandAugment: Cubuk et al. (2020), aplica num_ops transformaciones aleatorias con magnitud fija
# TrivialAugmentWide: Muller & Hutter (2021), sin busqueda de hiperparametros

def apply_rand_trivial_augment(image):
    """Aplica RandAugment y TrivialAugmentWide optimizados, empaquetando en un diccionario."""
    results = {}
    from torchvision import transforms
    from PIL import Image

    img_pil_input = Image.fromarray(image)

    # Aplicamos las transformaciones directamente sobre el objeto PIL
    rand_aug = transforms.RandAugment(num_ops=2, magnitude=9)
    trivial_aug = transforms.TrivialAugmentWide()

    for i in range(4):
        results[f'10_rand_augment_{i+1}'] = np.array(rand_aug(img_pil_input))
        results[f'10_trivial_augment_{i+1}'] = np.array(trivial_aug(img_pil_input))

    return results

# Ejecución local para renderizar el output visual en el notebook
smart_aug_results = apply_rand_trivial_augment(img_rgb)

rand_variantes = [smart_aug_results[f'10_rand_augment_{i+1}'] for i in range(4)]
trivial_variantes = [smart_aug_results[f'10_trivial_augment_{i+1}'] for i in range(4)]

print('RandAugment (num_ops=2, magnitude=9):')
mostrar([img_rgb] + rand_variantes, ['Original'] + [f'RandAugment {i+1}' for i in range(4)])

print('TrivialAugmentWide:')
mostrar([img_rgb] + trivial_variantes, ['Original'] + [f'TrivialAugment {i+1}' for i in range(4)])

### 11. AugMix (robustez ante corrupciones)

In [ ]:
# 11. AugMix
# Hendrycks et al. (2020): mezcla de multiples cadenas de aumento para robustez ante corrupciones
# Disponible en torchvision >= 0.13

def apply_augmix(image):
    """Aplica la política AugMix optimizada y empaqueta en un diccionario."""
    results = {}
    from torchvision import transforms
    from PIL import Image

    img_pil_input = Image.fromarray(image)

    # Aplicamos AugMix directamente sobre el objeto PIL
    aug_mix = transforms.AugMix(severity=3, mixture_width=3)

    for i in range(5):
        results[f'11_augmix_{i+1}'] = np.array(aug_mix(img_pil_input))

    return results

# Ejecución local para renderizar el output visual en el notebook
augmix_results = apply_augmix(img_rgb)

augmix_variantes = [augmix_results[f'11_augmix_{i+1}'] for i in range(5)]

mostrar(
    [img_rgb] + augmix_variantes,
    ['Original'] + [f'AugMix {i+1}' for i in range(5)]
)

In [ ]:
# ==============================================================================
# ORQUESTADOR BATCH: PIPELINE UNIFICADO Y ESCRITURA DIRECTA EN LARAGON
# ==============================================================================
import time
import shutil

def run_batch_augmentation_pipeline(images_list):
    print(f"\nIniciando procesamiento por lotes. Destino: {OUTPUT_DIR}")
    start_time = time.time()
    
    total_generated = 0
    
    for idx, (filename, img_array) in enumerate(images_list):
        print(f"-> Procesando imagen {idx + 1}/{len(images_list)}: {filename}")
        
        # Diccionario temporal para consolidar transformaciones de la imagen actual
        temp_results = {}
        
        # Ejecutamos transformaciones eficientes (Evitamos GAN/VGG19 para agilizar el batch en CPU)
        temp_results.update(apply_geometric_transformations(img_array))
        temp_results.update(apply_kernel_filters(img_array))
        temp_results.update(apply_color_space(img_array))
        temp_results.update(apply_random_erasing(img_array))
        temp_results.update(apply_autoaugment(img_array))
        temp_results.update(apply_augmix(img_array))
        
        # Truncar el diccionario para tomar exactamente 10 variaciones por imagen
        selected_results = dict(list(temp_results.items())[:10])
        
        # Operaciones de I/O: Guardar directamente en Laragon
        for aug_name, img_matrix in selected_results.items():
            # Casteo para OpenCV
            img_bgr = cv2.cvtColor(img_matrix, cv2.COLOR_RGB2BGR)
            
            # Nomenclatura: nombreOriginal_nombreTransformacion.jpg
            final_name = f"{filename}_{aug_name}.jpg"
            filepath = os.path.join(OUTPUT_DIR, final_name)
            cv2.imwrite(filepath, img_bgr)
            
            total_generated += 1
            
    end_time = time.time()
    execution_time = end_time - start_time
    
    print("-" * 50)
    print(f" Pipeline Batch finalizado con éxito.")
    print(f" Total de imágenes generadas: {total_generated}")
    print(f" Tiempo total de ejecución (CPU + I/O): {execution_time:.4f} segundos")
    print("-" * 50)

# Trigger del orquestador por lotes
if input_images:
    run_batch_augmentation_pipeline(input_images)
else:
    print("No se seleccionaron imágenes en el paso anterior.")

---
## Resumen de tecnicas cubiertas

| Categoria | Tecnica | Libreria |
|---|---|---|
| Basic - Geometric | Flip, Rotate, Scale, Perspective, Elastic | albumentations |
| Basic - Kernel | Gaussian Blur, Sharpen, Sobel, Laplacian, Emboss | OpenCV |
| Basic - Color Space | Brightness, Contrast, Hue-Sat, CLAHE, Gamma | albumentations |
| Basic - Random Erasing | Cutout, CoarseDropout, GridDropout | albumentations |
| Basic - Mixing | MixUp, CutMix | NumPy |
| DL - Adversarial | FGSM | PyTorch + ResNet18 |
| DL - Style Transfer | NST con VGG19 + Gram Matrix | PyTorch + VGG19 |
| DL - GAN | DCGAN entrenada en patches | PyTorch |
| Meta - AutoAugment | AutoAugment (ImageNet, CIFAR-10, SVHN) | torchvision |
| Meta - Smart | RandAugment, TrivialAugmentWide, AugMix | torchvision |

### Referencias
- Shorten & Khoshgoftaar (2019). A survey on Image Data Augmentation for Deep Learning. *Journal of Big Data*.
- Cubuk et al. (2019). AutoAugment: Learning Augmentation Strategies From Data. *CVPR*.
- Cubuk et al. (2020). RandAugment: Practical automated data augmentation. *NeurIPS*.
- Goodfellow et al. (2014). Generative Adversarial Networks. *NeurIPS*.
- Gatys et al. (2015). A Neural Algorithm of Artistic Style. *arXiv*.
- Yun et al. (2019). CutMix: Training Strategy that Makes Strong Classifiers. *ICCV*.
- Zhang et al. (2018). MixUp: Beyond Empirical Risk Minimization. *ICLR*.
- Hendrycks et al. (2020). AugMix: A Simple Data Processing Method to Improve Robustness. *ICLR*.